# 01 · 메인 학습 — aloha insertion (150k / seed 4개)

**프로토콜 (확정)**
- **150k step 까지만** 학습. 그 다음 `02_eval_main` 이 **150k 체크포인트**를 5회 반복 평가.
- **seed 4개** (`cf.MAIN_SEEDS = [0,1,2,3]`). 은지와 나눠 돌릴 땐 아래 `SEEDS` 만 바꾸면 됨 (결과는 자동 pooled).
- **lr sweep 없음.** 전 모델 고정 1e-5 (diffusion/smolvla 만 원 논문 1e-4).

**무엇을 돌릴지 = `TAGS` 프리셋으로 선택** (GPU 예산 문제)

| 프리셋 | 태그 | 잡 수 | 언제 |
|---|---|---|---|
| `cf.TRAIN_OURS` | `ours` | **4** | 우리 모델만 빨리 |
| `cf.TRAIN_OURS_CTRL` ★ | `acm` + `ours` | **8** | **최소 권장** — "acm 대비 개선"이 우리 헤드라인 주장이라 대조군 `acm` 은 사실상 필수 |
| `cf.TRAIN_ABLATION` | `acm_carry`·`acm_bimamba`·`acm_s7` | 12 | 기여 분해(논문 필수, `05` 에서 돌려도 됨) |
| `cf.TRAIN_BASELINE` | `act`·`diffusion`·`smolvla`·`acm2` | 16 | 메인 표 baseline |
| `cf.TRAIN_ALL` | 6모델 전부 | 24 | 한 번에 |

⚠️ **baseline 을 옛 런에서 가져다 쓰면 안 됨** — lr/step/seed 가 다르면 비교가 깨짐(리뷰어가 가장 먼저 보는 지점).
언젠가는 같은 프로토콜로 돌려야 함. 지금 안 돌릴 뿐.

⚠️ 학습 전에 **parity 테스트**: `00_smoke` 또는 `python tests/test_acm_sscp_literal.py`.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

TASK  = cf.MAIN_SIM          # 'insertion'
SEEDS = cf.MAIN_SEEDS        # [0,1,2,3]   (은지와 분담: 해준 [0,1] / 은지 [2,3])
NGPU  = 8

# ── 무엇을 돌릴지 ───────────────────────────────────────────────────────────
TAGS = cf.TRAIN_OURS         # ← 우리 모델만 (4잡)
# TAGS = cf.TRAIN_OURS_CTRL  #   + acm 대조군 (8잡) ★권장
# TAGS = cf.TRAIN_ABLATION   #   ablation 사다리 (12잡)
# TAGS = cf.TRAIN_BASELINE   #   외부 baseline (16잡)
# TAGS = cf.TRAIN_ALL        #   전부 (24잡)

print('task :', TASK, cf.v23.TASKS[TASK], '| fps', cf.fps_of(TASK))
print('steps:', f'{cf.STEPS:,}', '| seeds:', SEEDS, '| NGPU:', NGPU)
print('학습 :', TAGS, f'-> {len(TAGS) * len(SEEDS)} 잡')
print()
for t in TAGS:
    pol, lr, K, extra, cp = cf.v23.MODEL_CONFIGS[t]
    print(f'  {t:<12} {pol:<34} lr={lr:<7} K={K:<4} pairs={cp}')

## 커맨드 확인 (dry-run) — lr / steps / 플래그

In [ ]:
for t in TAGS:
    c = cf.make_train_cmd(t, seed=SEEDS[0], task=TASK, gpu_id=0)
    flags = [p for p in c.split() if p.startswith(('--policy.optimizer_lr', '--steps',
                                                   '--policy.chunk_size', '--policy.horizon',
                                                   '--use_chunk_pairs'))]
    print(f'{t:<12} ' + '  '.join(flags))
print()
print(cf.make_train_cmd(TAGS[-1], seed=SEEDS[0], task=TASK, gpu_id=0))

## 학습 — NGPU 만큼 청크로 (각 청크 끝날 때까지 대기, resume 자동)

In [ ]:
jobs = cf.train_jobs(SEEDS, tags=TAGS, task=TASK)
print('총', len(jobs), '잡')
for i in range(0, len(jobs), NGPU):
    chunk = jobs[i:i + NGPU]
    print('\n===== 청크 %d/%d (%d 잡) =====' % (i // NGPU + 1, -(-len(jobs) // NGPU), len(chunk)))
    for j in chunk:
        print('  ', j)
    cf.launch_training_live(chunk)
print('\n학습 완료 (150k):', TAGS, SEEDS)

## 상태 — 150k 도달 / 체크포인트 확인

In [ ]:
cf.print_training_status(jobs)
print()
print('150k 체크포인트:')
for s in SEEDS:
    row = []
    for t in TAGS:
        cd = cf.v23.best_ckpt_dir(t, s, TASK, how=cf.CKPT_STEP)
        if cd is None:
            row.append(f'{t}:X')
        else:
            step = int(cd.name)
            row.append(f'{t}:{step // 1000}k' + ('' if step == cf.CKPT_STEP else '(!)'))
    print(f'  seed{s}  ' + '  '.join(row))
print('\n다음: 02_eval_main (150k ckpt x 5회 반복 eval)')